In [6]:
# Run in conda ibiz environment
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import os
import pyproj
from scipy.spatial import cKDTree

# importlib seems to clutter the kernel

# CONFIGS
from preprocess_radargrams_configs import PATH_2016_AN_UTIG_ER2HI1B, PATH_2016_AN_UTIG_ER2HI2, CODE_1B, CODE_2
from preprocess_radargrams_configs import MATCHING_TOLERANCE, ROWS_CLIPPED_BLANKING, C, EPSILON_ICE
from preprocess_radargrams_configs import AMPLITUDE_VMIN, AMPLITUDE_VMAX, COLOR_MAP
from preprocess_radargrams_configs import IMAGE_TILE_WIDTH, IMAGE_TILE_HEIGHT

RUN_CHECKS_BOOL = True

In [7]:
# NOTE: Start with 2016 data, as it is more recent and has better coverage

# Go one level deeper into the tar file to get the folder containing the .nc files
PATH_2016_AN_UTIG_ER2HI1B_folder = os.path.join(PATH_2016_AN_UTIG_ER2HI1B, "2016_AN_UTIG.ER2HI1B")

# Extract the list of .nc files in the folderconda activate b
list_of_nc_files_2016_UTIG = [
    f for f in os.listdir(PATH_2016_AN_UTIG_ER2HI1B_folder) 
    if f.endswith(".nc")
    ]

list_of_nc_files_2016_UTIG_full_paths = [
    os.path.join(PATH_2016_AN_UTIG_ER2HI1B_folder, f)
    for f in os.listdir(PATH_2016_AN_UTIG_ER2HI1B_folder)
    if f.endswith(".nc")
]

print(f"Number of .nc files (granules) in 2016 AN UTIG ER2HI1B folder: {len(list_of_nc_files_2016_UTIG)}")

Number of .nc files (granules) in 2016 AN UTIG ER2HI1B folder: 237


# Proprocessing pipeline for radar data

## Steps
- Start from one image first
    - **high gain** more relevant for bed detection, **low gain** more relevant for surface detection
- Convert to depth
- Extract location of bed 
- crop IBIZ 
- Have a fixed scale
- Combine with bed picks

# Acronyms
- ER2HI1B - ER: EAGLE Radar (?!) - HI: HiCARS - 1B: Level 1B data
- ER2HI2 - ER: EAGLE Radar (?!) - HI: HiCARS - 2: Level 2 data
    - 2016 refers to the 2016/17 season so Jan 2017 is part of it
    - files are considered granules

In [8]:
torch.empty((1, 1, IMAGE_TILE_HEIGHT, IMAGE_TILE_WIDTH))

tensor([[[[0.0000e+00, 0.0000e+00, 2.5987e-36,  ..., 4.2326e+21,
           1.9349e-19, 4.4644e+30],
          [2.0706e-19, 1.1022e+24, 3.0346e+32,  ..., 3.6713e+03,
           4.7427e+30, 1.9517e-19],
          [4.5450e+30, 4.8561e+33, 1.7438e+28,  ..., 8.2990e-33,
           1.3563e-19, 1.3563e-19],
          ...,
          [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
           0.0000e+00, 0.0000e+00]]]])

In [ ]:
# Define before loop to avoid re-creating the transformer for each granule
HALF_TILE_WIDTH = IMAGE_TILE_WIDTH // 2
HALF_TILE_HEIGHT = IMAGE_TILE_HEIGHT // 2

lonlat_to_polarstereo = pyproj.Transformer.from_crs(crs_from = pyproj.CRS("epsg:4326"),
                                                    crs_to = pyproj.CRS("epsg:3031"),
                                                    always_xy = True) # xy order convention

# Containers
# convention: (1, 1, H, W) for PyTorch, where H is height and W is width
tiles_tensor = torch.empty((0, 1, IMAGE_TILE_HEIGHT, IMAGE_TILE_WIDTH), dtype = torch.float64)
tiles_metadata = pd.DataFrame(columns = ["survey_id", "granule_id", "x_idx", "X", "Y", "LON", "LAT", "bed_elevation", "surface_elevation", "ice_thickness", "partial_bed_reflect", "srf_reflect", "tile_along_track_span_m", "tile_vertical_span_m"])

for path_to_granule in list_of_nc_files_2016_UTIG_full_paths[0:100]:

    ###############
    # 1. L1B data: Open nc file as xarray dataset
    radargram_ds = xr.open_dataset(path_to_granule)
    print("------------------------------")
    print("Now processing Granule ID:", radargram_ds.granule_id)
    print("L1B radargram shape (rows, columns):", radargram_ds["amplitude_high_gain"].T.shape)

    # Sanity check
    columns_without_lonlat = (radargram_ds["lon"].isnull() | radargram_ds["lat"].isnull()).sum().item()
    print(f"L1B radargram number of columns missing LON / LAT / BOTH: {columns_without_lonlat}")

    # If there are any columns without lon/lat, skip this granule
    if columns_without_lonlat > 0:
        # NOTE: Often it is just the last 11/12 columns missing so we can "save" the granule
        radargram_ds_cropped = radargram_ds.isel({"time": slice(0, - columns_without_lonlat)})
        columns_without_lonlat_after_crop = (radargram_ds_cropped["lon"].isnull() | radargram_ds_cropped["lat"].isnull()).sum().item()
        if columns_without_lonlat_after_crop == 0:
            radargram_ds = radargram_ds_cropped
            print(f"INFO: Cropped {columns_without_lonlat} columns from the end of the radargram to remove missing LON / LAT values")
        else:
            print(f"WARNING: Skipping granule {radargram_ds.granule_id} due to missing lon/lat values")
            continue

    ###############
    # 2. L2 data: Open corresponding bed elevation file as pandas dataframe
    # Replace 1B with 2 in the granule ID to get the corresponding L2 granule ID
    radargram_grandule_id_L2 = radargram_ds.granule_id[:5] + "2" + radargram_ds.granule_id[7:]
    # Remove the last three characters (such as 000, 001 or 002) because L2 data filed exists per survey id, not per granule id
    # ends with _
    radargram_survey_id_L2 = radargram_grandule_id_L2[:-3]
    # print("L2 survey ID:", radargram_survey_id_L2)
    # NOTE: Hardcoded currently
    PATH_L2 = PATH_2016_AN_UTIG_ER2HI2 + "/" + radargram_survey_id_L2 + "icethk.txt"
    print("L2 file path:", PATH_L2)

    # Check if the L2 file exists
    if not os.path.exists(PATH_L2):
        print(f"WARNING: L2 file {PATH_L2} does not exist")
        continue

    # Load header lines (starting with #) from the L2 file first
    with open(PATH_L2, "r") as f:
        header_lines = [line for line in f if line.startswith("#")]

    # grab last header line for the column headers
    col_names = header_lines[-1].lstrip("#").split()

    # now import the data into a pandas dataframe, skipping the header lines, but adding the column names from the last header line
    l2_df_raw = pd.read_csv(
        PATH_L2,
        sep = r"\s+", 
        # skips any line starting with '#'
        comment = "#", 
        header = None, 
        names = col_names,
    )

    # SANITY CHECK: Completely remove rows with NaN values in the LON or LAT as they are not useful for our purposes
    print(f"L2 dataframe shape before dropping rows with NaN in LON or LAT: {l2_df_raw.shape[0]}")
    # Drop them so the the lookup will just not return a match for those rows, instead of returning a match with NaN values.
    l2_df = l2_df_raw.dropna(subset = ["LON", "LAT"]) 
    print(f"L2 dataframe shape after dropping rows with NaN in LON or LAT: {l2_df.shape[0]}")

    # if the L2 dataframe has no valid rows after dropping NaN values, skip this granule
    if l2_df.shape[0] == 0:
        print(f"WARNING: L2 dataframe {PATH_L2} has no valid rows after dropping NaN in LON or LAT")
        continue

    ###############
    # 3. Convert LON, LAT to X, Y coordinates in Polar Stereographic projection (EPSG:3031)
    # Pass LON and LAT columns, returns X and Y columns.
    x_array, y_array = lonlat_to_polarstereo.transform(l2_df["LON"], l2_df["LAT"])

    # Add the X and Y columns to the L2 dataframe
    l2_df["X"] = x_array
    l2_df["Y"] = y_array

    ###############
    # 4. Join L2 data onto L1B data
    # KCTree worked the best here because there are some minor inconsistencies between the L1B and L2 data (e.g. float32 and float64 coordinates), so we need to find the nearest neighbor in the L2 data for each L1B trace.
    # Make dtype consistent for both: radargram coords are originally float32, but L2 coords are float64, so we need to convert them to float32 for the KDTree to work properly.
    l1b_lon = radargram_ds["lon"].values.astype(np.float32)
    l1b_lat = radargram_ds["lat"].values.astype(np.float32)

    l2_lon = l2_df["LON"].values.astype(np.float32)
    l2_lat = l2_df["LAT"].values.astype(np.float32)

    # stack
    l1b_coords = np.column_stack([l1b_lon, l1b_lat])
    l2_coords = np.column_stack([l2_lon, l2_lat])

    # build tree on l2_df (the side we're looking up / pulling matches from)
    tree = cKDTree(l2_coords)

    MATCHING_TOLERANCE = 2e-5  # degrees, ~100 m
    distances, indices = tree.query(l1b_coords, distance_upper_bound = MATCHING_TOLERANCE)

    # boolean indicator of whether each L1B trace has a match in the L2 data within the matching tolerance
    has_match = distances < MATCHING_TOLERANCE

    # where has_match is True, use the index from indices, otherwise use 0 to avoid indexing errors
    safe_indices = np.where(has_match, indices, 0)  # placeholder for unmatched rows

    matched_l2_subset = l2_df.iloc[safe_indices].reset_index(drop = True)

    # fill rows that weren't matched with NaN
    matched_l2_subset.loc[~ has_match] = np.nan

    # Communicate how many matches were found
    print(f"Matched: {has_match.sum()} / {len(l1b_coords)}")
    continue

    # select columns that will be added to the radargram xarray dataset
    selected_cols = ["THK", "BED_ELEVATION", "SURFACE_ELEVATION", "PARTIAL_BED_REFLECT", "SRF_REFLECT", "SRF_RNG", "X", "Y"]

    # Add the selected columns from the matched L2 subset to the radargram xarray dataset
    for col in selected_cols:
        # Add along time dimension in xarray
        radargram_ds[f"l2_{col}"] = ("time", matched_l2_subset[col].values)

    # free-up memory
    del tree

    # General consistency check:
    if RUN_CHECKS_BOOL == True:
        residual = radargram_ds["altitude"] - radargram_ds["l2_SRF_RNG"] - radargram_ds["l2_SURFACE_ELEVATION"]
        print(f"Maximum residual: {np.max(np.abs(residual)).item():.2f}")

    ###############
    # 5. Generate 2D elevation field

    # Calculate refractive index of ice
    n = np.sqrt(EPSILON_ICE)
    # Calculate speed of light in ice
    v_ice = C / n 

    ### AIR GAP HANDLING ###
    # air gap = plane altitude above surface = one-way distance from antenna to surface, in meters
    # NOTE: Alternatively use surface range (SRF_RNG) from L2 data, directly
    air_gap_m = radargram_ds.altitude.values - radargram_ds.l2_SURFACE_ELEVATION.values

    # Two-way travel time spent in air (round trip through air only, given air gap in meters (GIVEN) and speed of light in vacuum)
    air_twtt_seconds = (2 * air_gap_m) / C

    # Convert fasttime (TWTT, in microseconds) from y-axis to seconds
    fasttime_in_seconds = radargram_ds.fasttime.values * 1e-6

    ### ICE HANDLING ###
    # Subtract the air portion, leaving only the ice portion of the TWTT
    # fasttime_in_seconds is a column vector (y-axis), air_twtt_seconds is a row vector (x-axis), so we can broadcast the subtraction to get a 2D array of ice TWTT values
    # NOTE: negative values mean time still in air, so we can mask those out later if needed
    ice_twtt_seconds = fasttime_in_seconds.ravel()[:, np.newaxis] - air_twtt_seconds.ravel()[np.newaxis, :]

    # mask is true where sample is still in air (before surface return)
    in_air_mask = ice_twtt_seconds < 0

    # CONTAINER 2d array
    one_way_distance_m = np.empty_like(ice_twtt_seconds)

    ### FILL AIR CELLS given air resistance (speed of light in air) ###
    one_way_distance_m[in_air_mask] = C * fasttime_in_seconds[:, np.newaxis].repeat(ice_twtt_seconds.shape[1], axis = 1)[in_air_mask] / 2

    ### FILL ICE CELLS given ice resistance (speed of light in ice) ###
    one_way_distance_m[ ~ in_air_mask] = (air_gap_m[np.newaxis, :] + v_ice * ice_twtt_seconds / 2)[ ~ in_air_mask]

    # Get elevation above sea level (m) for column of the radargram, by subtracting the one-way distance from the plane altitude
    # This makes it Absolute
    elevation_m = radargram_ds.altitude.values[None, :] - one_way_distance_m

    ###############
    # 6. OFFSET BLANKING: Remove the first ROWS_CLIPPED_BLANKING rows of the radargram, which are blanked out due to the radar system's internal delay
    # bed idx includes the offset

    # Retrieve interpreted bed elevation values (in meters above sea level)
    # shape (4000,)
    # NOTE: some of the bed elevation values are NaN, so we need to handle that
    bed_elevation = radargram_ds.l2_BED_ELEVATION.values 

    # For each column, find the row index where elevation_m is closest to bed_elevation
    # NOTE: we need floats to allow for NaN values, so we cast to float
    bed_idx_no_offset = np.argmin(np.abs(elevation_m - bed_elevation[np.newaxis, :]), axis = 0).astype(float)

    # Propagate NaN where bed_elevation itself was NaN
    bed_idx_no_offset[np.isnan(bed_elevation)] = np.nan

    # ADD THE OFFSET for the clipped blanking rows to get the final bed index
    bed_idx = bed_idx_no_offset + ROWS_CLIPPED_BLANKING
    # print("bed_idx shape:", bed_idx.shape) 

    ###############
    # 7. CROP tiles
    # Extract and transpose high gain amplitude 2D array 
    amplitude = radargram_ds["amplitude_high_gain"].values.T 

    # Loop through x axis indices that result in a full tile
    for x_idx in range(HALF_TILE_WIDTH, amplitude.shape[1] - HALF_TILE_WIDTH, 1):
        x_center_index = x_idx

        # Calculate min and max indices
        x_min_index = x_center_index - HALF_TILE_WIDTH
        x_max_index = x_center_index + HALF_TILE_WIDTH

        # Retrieve Y CENTER INDEX from bed_idx for the current x_center_index
        y_center_index = bed_idx[x_center_index].item()

        # Skip the current iteration if y_center_index is NaN (no valid bed pick)
        if np.isnan(y_center_index):
            print(f"Skipping x_idx {x_idx}: no valid bed pick")
            # Skips rest of loop
            continue

        # NOTE: Needs to be an integer for indexing, but bed_idx can be float due to NaN handling
        y_center_index = int(y_center_index)

        # Calculate min and max indices
        y_min_index = y_center_index - HALF_TILE_HEIGHT 
        y_max_index = y_center_index + HALF_TILE_HEIGHT

        # NOTE: Check vertical range
        if y_min_index < 0 or y_max_index >= amplitude.shape[0]:
            print(f"Skipping x_idx {x_idx}: y indices out of bounds")
            continue

        # Calculate some metrics
        x_min_ps, x_max_ps, y_min_ps, y_max_ps = radargram_ds.l2_X[x_min_index, ].values, radargram_ds.l2_X[x_max_index, ].values, radargram_ds.l2_Y[y_min_index, ].values, radargram_ds.l2_Y[y_max_index, ].values
        along_track_span = np.sqrt((x_max_ps - x_min_ps)**2 + (y_max_ps - y_min_ps)**2)
        # along track span in meters. e.g. 336 pixels * 21 m/pixel = 7 km
        print(f"along_track_span in meters: {along_track_span:.1f}")

        # Indicate vertical span of the tile for the (x axis) center of the tile
        vertical_span = elevation_m[y_min_index, x_center_index] - elevation_m[y_max_index, x_center_index]
        print(f"vertical_span in meters: {vertical_span:.1f}")

        # Extract the tile from the amplitude array
        tile = amplitude[y_min_index : y_max_index, x_min_index : x_max_index]
        print("tile shape:", tile.shape)

        # Convert the tile to a PyTorch tensor and add a batch dimension (1, 1, H, W)
        tile_torch = torch.from_numpy(tile).unsqueeze(0).unsqueeze(0).to(torch.float64)  # (1, 1, 256, 256)
        tiles_tensor = torch.cat([tiles_tensor, tile_torch], dim = 0)
        # NOTE: could add low_amplitude tiles to a separate tensor, but for now we will just skip them

        # ADD METADATA
        tiles_metadata.loc[len(tiles_metadata)] = {
            "survey_id": radargram_survey_id_L2,
            "granule_id": radargram_ds.granule_id,
            "x_idx": x_idx,
            "X": radargram_ds.l2_X[x_center_index].item(),
            "Y": radargram_ds.l2_Y[x_center_index].item(),
            "LON": radargram_ds.lon.values[x_center_index],
            "LAT": radargram_ds.lat.values[x_center_index],
            "bed_elevation": radargram_ds.l2_BED_ELEVATION.values[x_center_index],
            "surface_elevation": radargram_ds.l2_SURFACE_ELEVATION.values[x_center_index],
            "ice_thickness": radargram_ds.l2_THK.values[x_center_index],
            "partial_bed_reflect": radargram_ds.l2_PARTIAL_BED_REFLECT.values[x_center_index],
            "srf_reflect": radargram_ds.l2_SRF_REFLECT.values[x_center_index],
            "tile_along_track_span_m": along_track_span,
            "tile_vertical_span_m": vertical_span,
        }
        break
    # break


------------------------------
Now processing Granule ID: ER2HI1B_2017024_TOT1_JKB2r_X19a_000
L1B radargram shape (rows, columns): (3200, 4000)
L1B radargram number of columns missing LON / LAT / BOTH: 0
L2 file path: /Users/kim.bente/Documents/ANT_QGIS/AADC-DATA/AAS_4346_EAGLE_ICECAP_LEVEL2_AEROGEOPHYSICS/EAGLE-2015-2016-Level2/Level2/2016_AN_UTIG.ER2HI2/ER2HI2_2017024_TOT1_JKB2r_X19a_icethk.txt
L2 dataframe shape before dropping rows with NaN in LON or LAT: 6408
L2 dataframe shape after dropping rows with NaN in LON or LAT: 6408
Matched: 4000 / 4000
------------------------------
Now processing Granule ID: ER2HI1B_2017027_SHK_JKB2r_Y44a_000
L1B radargram shape (rows, columns): (3200, 4000)
L1B radargram number of columns missing LON / LAT / BOTH: 0
L2 file path: /Users/kim.bente/Documents/ANT_QGIS/AADC-DATA/AAS_4346_EAGLE_ICECAP_LEVEL2_AEROGEOPHYSICS/EAGLE-2015-2016-Level2/Level2/2016_AN_UTIG.ER2HI2/ER2HI2_2017027_SHK_JKB2r_Y44a_icethk.txt
L2 dataframe shape before dropping rows with

In [27]:
radargram_ds.lat[-20:]

<xarray.DataArray 'lat' (time: 20)> Size: 80B
array([-67.013954, -67.01388 , -67.013794, -67.01372 , -67.01363 , -67.01355 ,
       -67.01347 , -67.01339 ,        nan,        nan,        nan,        nan,
              nan,        nan,        nan,        nan,        nan,        nan,
              nan,        nan], dtype=float32)
Coordinates:
  * time     (time) datetime64[ns] 160B 2017-01-24T06:16:35.074218750 ... 201...
Attributes:
    standard_name:  latitude
    units:          degrees_north
    long_name:      Latitude of sample
    axis:           Y

In [35]:
l1b_coords

array([[104.35452 , -66.384445],
       [104.354004, -66.384445],
       [104.353485, -66.38444 ],
       ...,
       [102.315025, -66.36211 ],
       [102.31452 , -66.36209 ],
       [102.314026, -66.36208 ]], shape=(4000, 2), dtype=float32)

In [40]:
l2_df["LON"].values

array([108.348048, 108.347542, 108.347049, ..., 101.736308, 101.735815,
       101.73532 ], shape=(6404,))

In [41]:
radargram_ds["lon"].values

array([115.50344, 115.50386, 115.50429, ...,       nan,       nan,
             nan], shape=(3257,), dtype=float32)

In [ ]:
    l1b_lon = radargram_ds["lon"].values.astype(np.float32)
    l1b_lat = radargram_ds["lat"].values.astype(np.float32)

    l2_lon = l2_df["LON"].values.astype(np.float32)
    l2_lat = l2_df["LAT"].values.astype(np.float32)

    # stack
    l1b_coords = np.column_stack([l1b_lon, l1b_lat])
    l2_coords = np.column_stack([l2_lon, l2_lat])

    # build tree on l2_df (the side we're looking up / pulling matches from)
    tree = cKDTree(l2_coords)

    MATCHING_TOLERANCE = 1e-5  # degrees, ~100 m
    distances, indices = tree.query(l1b_coords, distance_upper_bound = MATCHING_TOLERANCE)

    # boolean indicator of whether each L1B trace has a match in the L2 data within the matching tolerance
    has_match = distances < MATCHING_TOLERANCE

    # where has_match is True, use the index from indices, otherwise use 0 to avoid indexing errors
    safe_indices = np.where(has_match, indices, 0)  # placeholder for unmatched rows

    matched_l2_subset = l2_df.iloc[safe_indices].reset_index(drop = True)

    # fill rows that weren't matched with NaN
    matched_l2_subset.loc[~ has_match] = np.nan

    # Communicate how many matches were found
    print(f"Matched: {has_match.sum()} / {len(l1b_coords)}")

array([[115.61588287, -67.21559906],
       [115.61637878, -67.21560669],
       [115.61688232, -67.21562195],
       ...,
       [117.56933594, -67.26092529],
       [117.56983948, -67.26094818],
       [117.5703125 , -67.26096344]], shape=(4000, 2))

In [28]:
# tiles_metadata
tiles_tensor[0, 0, :, :].shape
tiles_tensor[1, 0, :, :]

tensor([[152.2970, 149.4890, 152.0530,  ..., 149.2040, 149.4570, 147.4950],
        [153.9220, 150.1140, 152.9560,  ..., 148.8850, 149.0470, 146.6940],
        [154.5630, 151.0120, 153.2200,  ..., 147.7570, 148.7400, 147.0570],
        ...,
        [171.0070, 169.2900, 168.2430,  ..., 171.8680, 175.4040, 168.7410],
        [171.5750, 169.2230, 168.4710,  ..., 171.4300, 174.7080, 170.1460],
        [171.1590, 169.3500, 169.6860,  ..., 170.1750, 174.1500, 171.0660]],
       dtype=torch.float64)

In [ ]:
path_to_a_radargram = list_of_nc_files_2016_UTIG_full_paths[0]

radargram_ds = xr.open_dataset(path_to_a_radargram)
print("Granule ID:", radargram_ds.granule_id)
print("Radargram shape:", radargram_ds["amplitude_high_gain"].shape)
radargram_ds

In [ ]:
# Visualize the radargram
fig, ax = plt.subplots(figsize = (12, 6))

pcm = ax.pcolormesh(
    # pcolormesh expects input C to be (nrows, ncols) so we have to transpose the data array
    # 400 top rows radargram_ds["amplitude_high_gain"].T[0:400, :],
    radargram_ds["amplitude_high_gain"].T[:, :],
    # global configs for color map and amplitude range
    cmap = COLOR_MAP,
    vmin = AMPLITUDE_VMIN,
    vmax = AMPLITUDE_VMAX
)

# radar convention: surface at top
# Now it is visualised like a table with the origin in the TOP LEFT corner
ax.invert_yaxis() 
plt.colorbar(pcm, ax = ax, label = "Amplitude")
plt.show()

# Find matching L2 data (bed picks)

Granule ID: ER2HI1B_2017024_TOT1_JKB2r_X19a_000

## Construct full path from info we already have 

In [ ]:
radargram_grandule_id_L2 = radargram_ds.granule_id[:5] + "2" + radargram_ds.granule_id[7:]
radargram_grandule_id_L2_cropped = radargram_grandule_id_L2[:-3]

PATH_L1B = PATH_2016_AN_UTIG_ER2HI2 + "/" + radargram_grandule_id_L2_cropped + "icethk.txt"
print(PATH_L1B)
# print("/Users/kim.bente/Documents/ANT_QGIS/AADC-DATA/AAS_4346_EAGLE_ICECAP_LEVEL2_AEROGEOPHYSICS/EAGLE-2015-2016-Level2/Level2/2016_AN_UTIG.ER2HI2/ER2HI2_2017024_TOT1_JKB2r_X19a_icethk.txt")

print()
# print the header lines from the L1B text file
with open(PATH_L1B, "r") as f:
    header_lines = [line for line in f if line.startswith("#")]

for line in header_lines:
    print(line, end = "")

In [ ]:
# grab last line for the column headers
col_names = header_lines[-1].lstrip("#").split()
print(col_names)

# now import the data into a pandas dataframe, skipping the header lines, but adding the column names from the last header line
l2_df = pd.read_csv(
    PATH_L1B,
    sep = r"\s+", 
    # skips any line starting with '#'
    comment = "#", 
    header = None, 
    names = col_names,
)

l2_df.head()

In [ ]:
# summary statistics
l2_df.describe()

## Convert to Polar Stereographic

- to calculate distance

In [ ]:
# This is how projections work. check through tool
lonlat_to_polarstereo = pyproj.Transformer.from_crs(crs_from = pyproj.CRS("epsg:4326"),
                                                    crs_to = pyproj.CRS("epsg:3031"),
                                                    always_xy = True) # xy order convention

# Pass lon and lat columns, returns x and y columns in polar stereographic coordinates
x_array, y_array = lonlat_to_polarstereo.transform(l2_df["LON"], l2_df["LAT"])

l2_df["X"] = x_array
l2_df["Y"] = y_array

# Calculate the distance between each point and the next point in the dataframe, using the X and Y columns
distance_to_next = ((l2_df["X"].shift(-1) - l2_df["X"])**2 + (l2_df["Y"].shift(-1) - l2_df["Y"])**2)**0.5
print("Distance to next point in meters:")
print(distance_to_next.describe())

# Join

Join additional info from L2 onto L1B. This may include:
- bed elevation (subtract bed elevation + flight altitude from L1B to get bed elevation)
    - unclear about surface range
- PARTIAL_BED_REFLECT # Field 10: Bed reflection coefficient @ 60 MHz (dB wrt perfect reflector; NO ICE LOSS ACCOUNTING)
- SRF_REFLECT # Field 11: Surface reflection coefficient @ 60 MHz (dB wrt perfect reflector)

Future:
- Use YEAR and DOY as alternative or additional way to match

# Match via LON LAT

- L1B data is float32
- L2 data is float64. When we simply convert L2 to float32 we match rather well. These are still some numerical issues but a tree and generate a 1:1 match.

In [ ]:
# Overlap of L2 coordinates and radargram coordinates
fig, ax = plt.subplots(figsize = (8, 8))


ax.scatter(l2_df["LON"].values, l2_df["LAT"].values, 
           s = 9, alpha = 0.1, label = "L2 file", c = l2_df.index, cmap = "viridis")
ax.scatter(radargram_ds["lon"].values, radargram_ds["lat"].values, 
           s = 1, alpha = 0.1, label = "radargram (L1B)", c = np.arange(radargram_ds.time.size), cmap = "YlOrRd_r")

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
ax.set_title("Radargram vs L2 coordinates")
plt.show()

# Join

In [ ]:
# Use consistent dtype (float64) for both — don't force float32
radar_lon = radargram_ds["lon"].values.astype(np.float64)
radar_lat = radargram_ds["lat"].values.astype(np.float64)

l2_lon = l2_df["LON"].values.astype(np.float64)
l2_lat = l2_df["LAT"].values.astype(np.float64)

radar_coords = np.column_stack([radar_lon, radar_lat])
l2_coords = np.column_stack([l2_lon, l2_lat])

# Build tree on l2_df (the side we're pulling matches from)
tree = cKDTree(l2_coords)
tol = 1e-5  # degrees; tune to your data's actual precision/spacing

distances, indices = tree.query(radar_coords, distance_upper_bound = tol)

# distances == inf / indices == len(l2_coords) means "no match found"
has_match = distances < tol

# indices is len(radargram_ds) long, aligned to radar order
# invalid indices point out-of-bounds, so clip/mask them before indexing l2_df
safe_indices = np.where(has_match, indices, 0)  # placeholder for unmatched rows

matched_l2_subset = l2_df.iloc[safe_indices].reset_index(drop = True)

# Null out rows that didn't actually match
matched_l2_subset.loc[~has_match] = np.nan

print(f"Matched: {has_match.sum()} / {len(radar_coords)}")

# Select columns I want to transfer
selected_cols = ["THK", "BED_ELEVATION", "SURFACE_ELEVATION", "PARTIAL_BED_REFLECT", "SRF_REFLECT", "SRF_RNG", "X", "Y"]

for col in selected_cols:
    # along time dim
    radargram_ds[f"l2_{col}"] = ("time", matched_l2_subset[col].values)

# free up memory
del tree

## Process L1B - Convert fasttime to meters

- Fasttime 3200 x Time 4000
- fasttime is 2-way travel time in µseconds
- Conversion differs for air and ice so do a dual approach
- every column will consequently have a sightly different converion of fasttime

In [ ]:
print("Fasttime units:")
print(radargram_ds.fasttime.long_name)
print()
print("Amplitude high gain units:")
print(radargram_ds.amplitude_high_gain.units)
print()
print(radargram_ds.amplitude_high_gain)
 # counts in dB

In [ ]:
# speed of light in vacuum
c = 2.998e8  # m/s

# relative permittivity (epsilon_r) of ice (~ 3.15 is standard for glacial ice)
epsilon_ice = 3.15

# refractive index of ice
n = np.sqrt(epsilon_ice)

# speed of light in ice
v_ice = c / n 

# --- Air gap handling ---
# "free air content" = one-way distance from antenna to surface, in meters
air_gap_m = radargram_ds.altitude.values - radargram_ds.l2_SURFACE_ELEVATION.values  # plane altitude above surface

# Two-way travel time spent in air (round trip through air only, given air gap in meters and speed of light in vacuum)
# Maybe 10% of time air travel
air_twtt_seconds = (2 * air_gap_m) / c

# Convert fasttime (TWTT, in microseconds) to seconds
fasttime_in_seconds = radargram_ds.fasttime.values * 1e-6

# Subtract the air portion, leaving only the ice portion of the TWTT
# fasttime_in_seconds is a column vector (y-axis), air_twtt_seconds is a row vector (x-axis), so we can broadcast the subtraction to get a 2D array of ice TWTT values
# NOTE: negative values mean time still in air, so we can mask those out later if needed
ice_twtt_seconds = fasttime_in_seconds.ravel()[:, np.newaxis] - air_twtt_seconds.ravel()[np.newaxis, :]

# --- Mask: True where sample is still in air (before surface return) ---
in_air_mask = ice_twtt_seconds < 0

# Container
one_way_distance_m = np.empty_like(ice_twtt_seconds)

# Fill air bins only: one way distance from altitude
one_way_distance_m[in_air_mask] = c * fasttime_in_seconds[:, np.newaxis].repeat(ice_twtt_seconds.shape[1], axis = 1)[in_air_mask] / 2

# Fill ice bins only
# Note: Maybe use cell atop
one_way_distance_m[~ in_air_mask] = (air_gap_m[np.newaxis, :] + v_ice * ice_twtt_seconds / 2)[~ in_air_mask]

# Get elevation above sea level (m) for each sample in the radargram, by subtracting the one-way distance from the plane altitude
elevation_m = radargram_ds.altitude.values[None, :] - one_way_distance_m

In [ ]:
# Consistency check:
residual = radargram_ds["altitude"] - radargram_ds["l2_SRF_RNG"] - radargram_ds["l2_SURFACE_ELEVATION"]
np.max(np.abs(residual))

In [ ]:
print(ROWS_CLIPPED_BLANKING)

in_air_mask_shifted = np.pad(
    in_air_mask,
    pad_width = ((ROWS_CLIPPED_BLANKING, 0), (0, 0)),  # pad ROWS_CLIPPED_BLANKING rows at top, 0 at bottom
    mode = "constant",
    constant_values = False
)[:-ROWS_CLIPPED_BLANKING, :]

print("in_air_mask shape:", in_air_mask.shape)

In [ ]:
fig, ax = plt.subplots(figsize = (12, 6))

# Transpose amplitude values to match pcolormesh's (nrows, ncols) convention
# amplitude shape: (3200, 4000) 
amplitude = radargram_ds["amplitude_high_gain"].values.T 

# this x axis is easier for indexing/zooming
fasttime_idx = np.arange(radargram_ds.fasttime.values.shape[0])                     
trace_idx = np.arange(amplitude.shape[1])          

mesh = ax.pcolormesh(trace_idx, fasttime_idx, amplitude, shading = "auto", cmap = COLOR_MAP, vmin = AMPLITUDE_VMIN, vmax = AMPLITUDE_VMAX)
fig.colorbar(mesh, ax = ax, label = "Amplitude (high gain, dB)")

# Overlay mask where we make 0's transparent (NaN) and 1's red
mask_overlay = np.where(in_air_mask_shifted, 1, np.nan) 
ax.pcolormesh(trace_idx, fasttime_idx, mask_overlay, shading = "auto", cmap = "Reds", alpha = 0.3, vmin = 0, vmax = 1)

ax.set_xlabel("Trace index (along track)")
ax.set_ylabel("Fasttime index (from surface)")
ax.set_title("Radargram with air-region mask overlay")

# Zoom by uncommenting this
ax.set_ylim(400, 600)
# ax.set_ylim(450, 500)

ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
print("in_air_mask shape:", in_air_mask.shape)
print("amplitude shape:", amplitude.shape)
print("Number of True (in-air) cells:", in_air_mask.sum())
print("Total cells:", in_air_mask.size)
print("Fraction in air:", in_air_mask.sum() / in_air_mask.size)

### Offset check

In [ ]:
fasttime_us = radargram_ds.fasttime.values  # shape (n_fasttime,)
dt_us = fasttime_us[1] - fasttime_us[0]      # sample spacing in microseconds
dt_s = dt_us * 1e-6

# Expected row index of the surface return, per trace
expected_surface_row = air_twtt_seconds / dt_s   # air_twtt_seconds already in seconds, shape (n_traces,)

# exclude high-amp first rows
amp = radargram_ds.amplitude_low_gain.values.T[350:, :]

# crude but effective: row of max amplitude per trace = surface return
observed_surface_row = np.argmax(amp, axis = 0)  # shape (n_traces,)
observed_surface_row
# 350 + 121 - 343 = 128

# Plot radargram with bed picks

In [ ]:
# 1. Retrieve interpreted bed elevation from L2 file (in meters above sea level)
# shape (4000,)
# NOTE: some of the bed elevation values are NaN, so we need to handle that
bed_elevation = radargram_ds.l2_BED_ELEVATION.values 

# 2. For each column, find the row index where elevation_m is closest to bed_elevation
# NOTE: we need floats to allow for NaN values, so we cast to float
bed_idx_no_offset = np.argmin(np.abs(elevation_m - bed_elevation[np.newaxis, :]), axis = 0).astype(float)

# Propagate NaN where bed_elevation itself was NaN
bed_idx_no_offset[np.isnan(bed_elevation)] = np.nan

# 3. Add the offset for the clipped blanking rows to get the final bed index
bed_idx = bed_idx_no_offset + ROWS_CLIPPED_BLANKING
print("bed_idx shape:", bed_idx.shape) 

# 4. Create a boolean mask for the bed locations
x_axis_index = np.arange(elevation_m.shape[1])
# container
bed_mask = np.zeros_like(elevation_m, dtype = bool)
# set to 1
# some nan values are shown as 0
bed_mask[bed_idx.astype(int), x_axis_index] = True

In [ ]:
fig, ax = plt.subplots(figsize = (12, 6))

# Transpose amplitude values to match pcolormesh's (nrows, ncols) convention
# amplitude shape: (3200, 4000) 
amplitude = radargram_ds["amplitude_high_gain"].values.T 

# this x axis is easier for indexing/zooming
fasttime_idx = np.arange(radargram_ds.fasttime.values.shape[0])                     
trace_idx = np.arange(amplitude.shape[1])          

mesh = ax.pcolormesh(trace_idx, fasttime_idx, amplitude, shading = "auto", cmap = COLOR_MAP, vmin = AMPLITUDE_VMIN, vmax = AMPLITUDE_VMAX)
fig.colorbar(mesh, ax = ax, label = "Amplitude (high gain, dB)")

# Overlay mask where we make 0's transparent (NaN) and 1's red
mask_overlay = np.where(bed_mask, 1, np.nan) 
ax.pcolormesh(trace_idx, fasttime_idx, mask_overlay, shading = "auto", cmap = "Reds", alpha = 1.0, vmin = 0, vmax = 1.5)

ax.set_xlabel("Trace index (along track)")
ax.set_ylabel("Fasttime index (from surface)")
ax.set_title("Radargram with bed mask overlay")

# Zoom by uncommenting this
# ax.set_ylim(1000, 1500)

ax.invert_yaxis()
plt.tight_layout()
plt.show()

# Crop tiles

- Naming: radar granule + x_axis index
- Along-track span
- Depth span

In [ ]:
IMAGE_TILE_WIDTH = 336
IMAGE_TILE_HEIGHT = IMAGE_TILE_WIDTH

HALF_TILE_WIDTH = IMAGE_TILE_WIDTH // 2
HALF_TILE_HEIGHT = IMAGE_TILE_HEIGHT // 2

PLOT_TILES_BOOL = False

for x_idx in range(HALF_TILE_WIDTH, amplitude.shape[1] - HALF_TILE_WIDTH, 1):
    print(x_idx)
    x_center_index = x_idx

    x_min_index = x_center_index - HALF_TILE_WIDTH
    print("x_min_index:", x_min_index)
    x_max_index = x_center_index + HALF_TILE_WIDTH
    print("x_max_index:", x_max_index)

    # Y CENTER INDEX
    y_center_index = bed_idx[x_center_index].item()
    
    if np.isnan(y_center_index):
        print(f"Skipping x_idx {x_idx}: no valid bed pick")
        # Skips rest of loop
        continue

    # NOTE: Needs to be an integer for indexing, but bed_idx can be float due to NaN handling
    y_center_index = int(y_center_index)

    # get indices for the tile
    y_min_index = y_center_index - HALF_TILE_HEIGHT 
    y_max_index = y_center_index + HALF_TILE_HEIGHT

    x_min_ps, x_max_ps, y_min_ps, y_max_ps = radargram_ds.l2_X[x_min_index, ].values, radargram_ds.l2_X[x_max_index, ].values, radargram_ds.l2_Y[y_min_index, ].values, radargram_ds.l2_Y[y_max_index, ].values
    along_track_span = np.sqrt((x_max_ps - x_min_ps)**2 + (y_max_ps - y_min_ps)**2)
    # along track span in meters. e.g. 336 pixels * 21 m/pixel = 7 km
    print(f"along_track_span in meters: {along_track_span:.1f}")

    # Indicate vertical span of the tile for the (x axis) center of the tile
    vertical_span = elevation_m[y_min_index, x_center_index] - elevation_m[y_max_index, x_center_index]
    print(f"vertical_span in meters: {vertical_span:.1f}")

    tile = amplitude[y_min_index : y_max_index, x_min_index : x_max_index]
    print("tile shape:", tile.shape)

    if PLOT_TILES_BOOL:
        fig, ax = plt.subplots(figsize = (8, 6))
        mesh = ax.pcolormesh(tile, shading = "auto", cmap = COLOR_MAP, vmin = AMPLITUDE_VMIN, vmax = AMPLITUDE_VMAX)

        # Mark the center (bed pick location)
        ax.plot(HALF_TILE_WIDTH, HALF_TILE_HEIGHT, marker = "+", color = "red", markersize = 15, markeredgewidth = 2)

        fig.colorbar(mesh, ax = ax, label = "Amplitude (dB)")
        ax.invert_yaxis()
        ax.set_aspect("equal")
        plt.show()

In [ ]:
x_idx = 1400
x_center_index = x_idx

x_min_index = x_center_index - HALF_TILE_WIDTH
print("x_min_index:", x_min_index)
x_max_index = x_center_index + HALF_TILE_WIDTH
print("x_max_index:", x_max_index)

# Y CENTER INDEX
y_center_index = bed_idx[x_center_index].item()

# NOTE: Needs to be an integer for indexing, but bed_idx can be float due to NaN handling
y_center_index = int(y_center_index)

# get indices for the tile
y_min_index = y_center_index - HALF_TILE_HEIGHT 
y_max_index = y_center_index + HALF_TILE_HEIGHT

x_min_ps, x_max_ps, y_min_ps, y_max_ps = radargram_ds.l2_X[x_min_index, ].values, radargram_ds.l2_X[x_max_index, ].values, radargram_ds.l2_Y[y_min_index, ].values, radargram_ds.l2_Y[y_max_index, ].values
along_track_span = np.sqrt((x_max_ps - x_min_ps)**2 + (y_max_ps - y_min_ps)**2)
# along track span in meters. e.g. 336 pixels * 21 m/pixel = 7 km
print(f"along_track_span in meters: {along_track_span:.1f}")

# Indicate vertical span of the tile for the (x axis) center of the tile
vertical_span = elevation_m[y_min_index, x_center_index] - elevation_m[y_max_index, x_center_index]
print(f"vertical_span in meters: {vertical_span:.1f}")

tile = amplitude[y_min_index : y_max_index, x_min_index : x_max_index]
print("tile shape:", tile.shape)

fig, ax = plt.subplots(figsize = (8, 6))
mesh = ax.pcolormesh(tile, shading = "auto", cmap = COLOR_MAP, vmin = AMPLITUDE_VMIN, vmax = AMPLITUDE_VMAX)

# Mark the center (bed pick location)
ax.plot(HALF_TILE_WIDTH, HALF_TILE_HEIGHT, marker = "+", color = "red", markersize = 15, markeredgewidth = 2)

fig.colorbar(mesh, ax = ax, label = "Amplitude (dB)")
ax.invert_yaxis()
ax.set_aspect("equal")
plt.show()

In [ ]:
# This is the more relatis aspect ratio for the tile, which is more like what the model will see
fig, ax = plt.subplots(figsize = (70, 5))
mesh = ax.pcolormesh(tile, shading = "auto", cmap = COLOR_MAP, vmin = AMPLITUDE_VMIN, vmax = AMPLITUDE_VMAX)

# Mark the center (bed pick location)
ax.plot(HALF_TILE_WIDTH, HALF_TILE_HEIGHT, marker = "+", color = "red", markersize = 15, markeredgewidth = 2)

fig.colorbar(mesh, ax = ax, label = "Amplitude (dB)")
ax.invert_yaxis()
plt.show()

# Questions:
- Offset choice
- Use reflectivity and other for correlation analysis (after) or also for processing?
- Build corresponding meta data tensor (index) and image dataset

# METADATA

- vertical span
- horizontal span
- relative reflectively
- lon lat
- x y

# 